n 2 n continued

In [65]:
import pandas as pd
import scipy.stats as stats

#check correlation btwn shots/goals and passes

url = "https://raw.githubusercontent.com/nkmwicz/worldcup2018data/refs/heads/main/cleaned_events_world_cup2018.csv"
df = pd.read_csv(url)
df.head()



,eventId,subEventName,tags,playerId,matchId,eventName,teamId,matchPeriod,eventSec,subEventId,id,x1,y1,x2,y2
0,8,Simple pass,['Accurate'],Mohammad Ibrahim Al Sahlawi,"Russia - Saudi Arabia, 5 - 0",Pass,Saudi Arabia,1H,1.656214,Simple pass,258612104,50,50,35.0,53.0
1,8,High pass,['Accurate'],Abdullah Ibrahim Otayf,"Russia - Saudi Arabia, 5 - 0",Pass,Saudi Arabia,1H,4.487814,High pass,258612106,35,53,75.0,19.0
2,1,Air duel,"['Won', 'Accurate']",Ilya Kutepov,"Russia - Saudi Arabia, 5 - 0",Duel,Russia,1H,5.937411,Air duel,258612077,25,81,37.0,83.0
3,1,Air duel,"['Lost', 'Not accurate']",Yasir Gharsan Al Shahrani,"Russia - Saudi Arabia, 5 - 0",Duel,Saudi Arabia,1H,6.406961,Air duel,258612112,75,19,63.0,17.0
4,8,Simple pass,['Accurate'],Salman Mohammed Al Faraj,"Russia - Saudi Arabia, 5 - 0",Pass,Saudi Arabia,1H,8.562167,Simple pass,258612110,63,17,71.0,15.0


In [66]:
from functions.eda import eda
eda(df)

=====DF Shape: (101759, 15)=====
***Numeric Columns***
eventId
	Num NA: 0
	% NA: 0.0%
	Min: 1
	Max: 10
	Range: 9
	25%: 1.0
	75%: 8.0
	St. Dev.: 3.115089378683579
	Num unique: 9
eventSec
	Num NA: 0
	% NA: 0.0%
	Min: 0.122552999999982
	Max: 3258.366837
	Range: 3258.244284
	25%: 643.4694855
	75%: 2115.7787175000003
	St. Dev.: 850.98050790337
	Num unique: 100135
id
	Num NA: 0
	% NA: 0.0%
	Min: 258612077
	Max: 280217520
	Range: 21605443
	25%: 259394664.5
	75%: 261094853.0
	St. Dev.: 4062609.8346007834
	Num unique: 101759
x1
	Num NA: 0
	% NA: 0.0%
	Min: 0
	Max: 100
	Range: 100
	25%: 31.0
	75%: 68.0
	St. Dev.: 24.560053845410334
	Num unique: 101
y1
	Num NA: 0
	% NA: 0.0%
	Min: 0
	Max: 101
	Range: 101
	25%: 23.0
	75%: 76.0
	St. Dev.: 30.01208708215375
	Num unique: 102
x2
	Num NA: 4
	% NA: 0.003930856238760208%
	Min: 0.0
	Max: 100.0
	Range: 100.0
	25%: 33.0
	75%: 71.0
	St. Dev.: 24.962312126892407
	Num unique: 101
y2
	Num NA: 4
	% NA: 0.003930856238760208%
	Min: 0.0
	Max: 101.0
	Range: 101.0
	2

In [67]:
#have to put passes and shots in list in the same order
    #could do per game, per team, or both
    #per team normalize
# we need to find all passes, shots, goals

events = df["eventName"].unique()
events

array(['Pass', 'Duel', 'Free Kick', 'Foul', 'Others on the ball', 'Shot',
       'Save attempt', 'Offside', 'Goalkeeper leaving line'], dtype=object)

In [68]:
df["pass"] = (df["eventName"] == "Pass").astype(int)
df["pass"].head()

0    1
1    1
2    0
3    0
4    1
Name: pass, dtype: int64

In [69]:
df["shot"] = (df["eventName"] == "Shot").astype(int)
df["shot"].head()

0    0
1    0
2    0
3    0
4    0
Name: shot, dtype: int64

In [70]:

df["subEventName"].unique()

array(['Simple pass', 'High pass', 'Air duel', 'Ground attacking duel',
       'Ground defending duel', 'Throw in', 'Foul', 'Free Kick',
       'Clearance', 'Touch', 'Ground loose ball duel', 'Corner',
       'Head pass', 'Launch', 'Acceleration', 'Shot', 'Smart pass',
       'Cross', 'Reflexes', nan, 'Hand pass', 'Goal kick',
       'Free kick cross', 'Hand foul', 'Save attempt', 'Free kick shot',
       'Goalkeeper leaving line', 'Penalty', 'Late card foul',
       'Violent Foul', 'Protest', 'Out of game foul', 'Time lost foul',
       'Simulation'], dtype=object)

In [71]:
import ast
df["tags"] = df["tags"].apply(ast.literal_eval)

In [72]:
unique_tags = list(set([tag for taglist in df["tags"] for tag in taglist]))
unique_tags.sort()
unique_tags

['Accurate',
 'Anticipated',
 'Anticipation',
 'Assist',
 'Blocked',
 'Counter attack',
 'Dangerous ball lost',
 'Direct',
 'Fairplay',
 'Feint',
 'Free space left',
 'Free space right',
 'Goal',
 'Head/body',
 'High',
 'Indirect',
 'Interception',
 'Key pass',
 'Left foot',
 'Lost',
 'Missed ball',
 'Neutral',
 'Not accurate',
 'Opportunity',
 'Own goal',
 'Position: Goal center',
 'Position: Goal center left',
 'Position: Goal center right',
 'Position: Goal high center',
 'Position: Goal high left',
 'Position: Goal high right',
 'Position: Goal low center',
 'Position: Goal low left',
 'Position: Goal low right',
 'Position: Out center left',
 'Position: Out center right',
 'Position: Out high center',
 'Position: Out high left',
 'Position: Out high right',
 'Position: Out low left',
 'Position: Out low right',
 'Position: Post center left',
 'Position: Post center right',
 'Position: Post high center',
 'Position: Post high left',
 'Position: Post high right',
 'Position: Post lo

In [73]:
df["goal"] = (
    (df["eventName"] == "Shot") & 
    (df["tags"].apply(lambda taglist: "Goal" in taglist))
    ).astype(int)

In [74]:
#could also do this
#df["goal"] = df.apply(lambda row: (row["eventName"] == "Shot") and ("Goal" in row["tags"]), axis=1).astype(int)

In [75]:
df["matchId"].nunique()

64

In [76]:
from typing import List
def group_data(columns: List[str]):
    grp = df.groupby(columns).agg(
        pass_count=("pass", "sum"),
        shot_count=("shot", "sum"),
        goal_count=("goal", "sum"),
        matches_count=("matchId", "nunique"),
    ).reset_index()
    return grp

## By MatchID

In [77]:
m_grp = group_data(["matchId"])
r, p = stats.pearsonr(m_grp["pass_count"], m_grp["shot_count"])
r2 = r**2
print(f"pass-shots: matchID: r={r:.4f}, p={p:.4f}, r2={r2:.4f}")

pass-shots: matchID: r=0.1996, p=0.1138, r2=0.0398


## By TeamID

In [80]:
m_grp = group_data(["teamId"])
r, p = stats.pearsonr(m_grp["pass_count"], m_grp["shot_count"])
r2 = r**2
print(f"pass-shots: matchID: r={r: .4f}, p={p: .4f}, r2={r2: .4f}")
r, p = stats.pearsonr(m_grp["pass_count"], m_grp["goal_count"])
r2 = r**2
print(f"pass-goals: matchID: r={r:.4f}, p={p:.4f}, r2={r2:.4f}")

pass-shots: matchID: r= 0.8932, p= 0.0000, r2= 0.7979
pass-goals: matchID: r=0.8541, p=0.0000, r2=0.7295


## By TeamID Normalized

In [81]:
m_grp = group_data(["teamId"])
m_grp["pass_norm"] = m_grp["pass_count"] / m_grp["matches_count"]
m_grp["shots_norm"] = m_grp["shot_count"] /m_grp["matches_count"]
m_grp["goals_norm"] = m_grp["goal_count"] / m_grp["matches_count"]

r, p = stats.pearsonr(m_grp["pass_norm"], m_grp["shots_norm"])
r2 = r**2
print(f"pass-shots: matchID: r={r:.4f}, p={p:.4f}, r2={r2:.4f}")
r, p = stats.pearsonr(m_grp["pass_norm"], m_grp["goals_norm"])
r2 = r**2
print(f"pass-goals: matchID: r={r:.4f}, p={p:.4f}, r2={r2:.4f}")

pass-shots: matchID: r=0.6667, p=0.0000, r2=0.4445
pass-goals: matchID: r=0.4271, p=0.0148, r2=0.1824


## By MatchID and TeamID

In [82]:
m_grp = group_data(["matchId", "teamId"])
r, p = stats.pearsonr(m_grp["pass_count"], m_grp["shot_count"])
r2 = r**2
print(f"pass-shots: matchID: r={r:.4f}, p={p:.4f}, r2={r2:.4f}")
r, p = stats.pearsonr(m_grp["pass_count"], m_grp["goal_count"])
r2 = r**2
print(f"pass-goals: matchID: r={r:.4f}, p={p:.4f}, r2={r2:.4f}")

pass-shots: matchID: r=0.5642, p=0.0000, r2=0.3184
pass-goals: matchID: r=0.1200, p=0.1772, r2=0.0144


## shots to goals

In [83]:
m_grp = group_data(["teamId"])
r, p = stats.pearsonr(m_grp["shot_count"], m_grp["goal_count"])
r2 = r**2
print(f"shots-goals: matchID: r={r:.4f}, p={p:.4f}, r2={r2:.4f}")

shots-goals: matchID: r=0.8404, p=0.0000, r2=0.7063


In [84]:
m_grp = group_data(["matchId","teamId"])
r, p = stats.pearsonr(m_grp["shot_count"], m_grp["goal_count"])
r2 = r**2
print(f"shots-goals:matchID & teamid: r={r:.4f}, p={p:.4f}, r2={r2:.4f}")

shots-goals:matchID & teamid: r=0.2107, p=0.0170, r2=0.0444
